In [14]:
import pandas as pd
import numpy as np
import os

file_path = "C:/Users/Maria/Downloads/data_dirty.csv"
df = pd.read_csv(file_path)

log = []
initial_rows = df.shape[0]

missing_before = df.isnull().sum().sum()
log.append(f"Missing values before: {missing_before}")

df = df.drop_duplicates()
after_duplicates = df.shape[0]
log.append(f"Duplicates removed: {initial_rows - after_duplicates}")

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip().str.lower()
log.append("Text columns standardized to lowercase and stripped spaces")

for col in df.columns:
    if df[col].dtype == 'object':
        try:
            df[col] = pd.to_numeric(df[col])
        except:
            pass
log.append("Attempted dtype corrections")

for col in df.select_dtypes(include=['int64', 'float64']).columns:
    df[col].fillna(df[col].median(), inplace=True)
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)
missing_after = df.isnull().sum().sum()
log.append(f"Missing values after: {missing_after}")

for col in df.select_dtypes(include=['int64', 'float64']).columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    df = df[(df[col] >= lower) & (df[col] <= upper)]
    log.append(f"Outliers removed from {col}: {outliers}")

final_rows = df.shape[0]

output_path = "C:/Users/Maria/Downloads/cleaned_data.csv"
df.to_csv(output_path, index=False)

print("Cleaning Log:")
for i in log:
    print(i)

print("Rows before:", initial_rows)
print("Rows after:", final_rows)
print(f"Cleaned file saved as {output_path}")

Cleaning Log:
Missing values before: 0
Duplicates removed: 167
Text columns standardized to lowercase and stripped spaces
Attempted dtype corrections
Missing values after: 0
Rows before: 1011
Rows after: 844
Cleaned file saved as C:/Users/Maria/Downloads/cleaned_data.csv
